# 06 — Monte Carlo: how likely is CNG to pay, and what matters most?

Notebook 02 gave three scenarios; notebook 03 gave one answer per surveyed driver. Here we combine the uncertainty
in **prices**, **who the driver is**, **queue time** and **loan terms**, and run the cost model 10,000 times. The
result is a *probability* that CNG pays off and a range of outcomes, not a single number.

All the sampling is in `src/monte_carlo.py`. Read its docstring for how each input is drawn. Inputs come from
`data/raw/model_inputs.csv` (sources in `docs/sources.md`) and the unflagged survey rows in
`data/processed/survey_clean.csv` (S47).

**What one draw is:** a randomly chosen surveyed driver (any of the 20 unflagged, petrol or CNG) converting to CNG,
at randomly drawn petrol and CNG prices, with a randomly drawn daily queue time and, in the market case, a
randomly drawn kit price and loan.

**Net benefit is per working day while the loan is being repaid.** After the loan (1–3 years), the market case
becomes the free case.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import monte_carlo as mc   # noqa: E402

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

N_DRAWS, SEED = 10_000, 42
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#ecebe7"
BLUE, ORANGE = "#2a78d6", "#eb6834"
FONT = dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK)

## 2. Run the simulation: market price and free conversion

Both runs use the same seed. `run_simulation` draws its random numbers in the same order either way, so each
"market" draw and its "free" twin have the same prices, driver and queue. Only the loan differs.

In [2]:
market = mc.run_simulation(N_DRAWS, SEED, conversion="market")
free = mc.run_simulation(N_DRAWS, SEED, conversion="free")

drivers = mc.load_drivers()
respondents = drivers["vehicle_type"].value_counts()
print(f"{N_DRAWS:,} draws each. Drivers in the bootstrap pool: {len(drivers)} "
      f"({drivers['uses_cng'].sum()} CNG, {(~drivers['uses_cng']).sum()} petrol)")
print(respondents.to_string())
print(f"\nMarket-case loan repayment: median NGN {market['loan_cost'].median():,.0f}/day "
      f"(10th-90th: {market['loan_cost'].quantile(0.1):,.0f}-{market['loan_cost'].quantile(0.9):,.0f})")

10,000 draws each. Drivers in the bootstrap pool: 20 (13 CNG, 7 petrol)
vehicle_type
danfo           8
ride_hailing    8
keke            3
taxi            1

Market-case loan repayment: median NGN 1,792/day (10th-90th: 705-2,736)


## 3. Probability that CNG pays

Each vehicle group's draws come from a handful of real drivers, re-used thousands of times. 10,000 draws can't create
information that isn't in the survey, so groups with **fewer than 5 respondents** are marked as insufficient data.
Their numbers are shown, but they are not results.

In [3]:
MIN_RESPONDENTS = 5


def summarise(result, label):
    # One row per group: pooled, then each vehicle type.
    groups = [("All drivers (pooled)", result)] + [
        (v, result[result["vehicle_type"] == v]) for v in ["ride_hailing", "danfo", "keke", "taxi"]
    ]
    rows = []
    for name, g in groups:
        n_resp = len(drivers) if name.startswith("All") else respondents.get(name, 0)
        nb = g["net_benefit"]
        rows.append({
            "group": name,
            "case": label,
            "respondents": n_resp,
            "P(CNG pays)": (nb > 0).mean(),
            "median NGN/day": nb.median(),
            "10th-90th pct NGN/day": f"{nb.quantile(0.1):,.0f} to {nb.quantile(0.9):,.0f}",
            "note": "" if n_resp >= MIN_RESPONDENTS else f"insufficient data ({n_resp} respondent{'' if n_resp == 1 else 's'})",
        })
    return pd.DataFrame(rows)


probability = pd.concat([summarise(market, "market"), summarise(free, "free")])
probability = probability.set_index(["group", "case"]).sort_index(level=0, sort_remaining=False)
order = ["All drivers (pooled)", "ride_hailing", "danfo", "keke", "taxi"]
probability = probability.reindex(order, level=0)
probability.style.format({"P(CNG pays)": "{:.0%}", "median NGN/day": "{:,.0f}"})

## 4. Distribution of net benefit (market case)

A histogram of all 10,000 draws. Everything to the right of the zero line is a draw where CNG pays.

In [4]:
nb = market["net_benefit"]
share_pays = (nb > 0).mean()

fig = go.Figure(go.Histogram(
    x=nb, xbins=dict(size=2_000), marker=dict(color=BLUE, line=dict(color="white", width=1)),
    hovertemplate="Net benefit ₦%{x}/day<br>%{y:,} draws<extra></extra>",
))
fig.add_vline(x=0, line=dict(color=INK, width=2))
fig.add_annotation(x=0, y=1, yref="paper", yanchor="bottom", xanchor="left", xshift=8, showarrow=False, align="left",
                   text=f"<b>CNG pays in {share_pays:.0%} of draws</b><br>median ₦{nb.median():,.0f}/day",
                   font=dict(color=INK, size=12))
fig.add_annotation(x=0, y=1, yref="paper", yanchor="bottom", xanchor="right", xshift=-8, showarrow=False, align="right",
                   text=f"CNG loses money<br>{1 - share_pays:.0%} of draws", font=dict(color=INK_MUTED, size=12))
fig.update_layout(
    title=dict(text="Net benefit of converting to CNG at market price<br>"
                    f"<sup>{N_DRAWS:,} Monte Carlo draws: survey drivers (S47), triangular prices and loan terms from "
                    "model_inputs.csv. ₦ per working day while repaying the loan.</sup>", x=0, xanchor="left"),
    template="plotly_white", font=FONT, bargap=0.02, showlegend=False,
    xaxis=dict(title="Net benefit (₦ per working day)", tickformat=",", gridcolor=GRID, zeroline=False),
    yaxis=dict(title="Number of draws", gridcolor=GRID, tickformat=","),
    hoverlabel=dict(bgcolor="white", font=dict(color=INK)),
    width=820, height=500, margin=dict(t=140, l=70, r=30, b=60),
)
fig.show()
fig.write_html(FIG_DIR / "monte_carlo_net_benefit.html", include_plotlyjs="cdn")
print("Saved outputs/figures/monte_carlo_net_benefit.html")

Saved outputs/figures/monte_carlo_net_benefit.html


## 5. Sensitivity: which input moves net benefit most?

**Spearman correlation** between each sampled input and net benefit, across the 10,000 market draws. Spearman ranks
the values first, so it captures "when this input is higher, is net benefit usually higher?" even if the
relationship isn't a straight line. The scale runs from −1 to +1: sign is direction, size is strength.

Driver-level inputs (fuel spend, earnings per hour, working days) are bootstrapped *together*, so their correlations
partly reflect each other. They show which inputs matter in this sample, not a clean cause and effect.

In [5]:
INPUT_LABELS = {
    "daily_queue_hours": "Daily queue time",
    "survey_fuel_spend": "Driver's petrol spend (at survey price)",
    "net_earnings_per_hour": "Driver's net earnings per hour",
    "petrol_price": "Petrol price",
    "conversion_cost": "Conversion cost",
    "cng_price": "CNG price",
    "interest_rate": "Loan interest rate",
    "working_days": "Working days per year",
    "tenor_years": "Loan tenor",
}
# pandas computes Spearman itself by ranking each column and correlating the ranks.
sensitivity = (market[list(INPUT_LABELS) + ["net_benefit"]]
               .corr(method="spearman")["net_benefit"].drop("net_benefit"))
sensitivity = sensitivity.reindex(sensitivity.abs().sort_values(ascending=False).index)
sensitivity.rename(index=INPUT_LABELS).to_frame("Spearman correlation with net benefit").style.format("{:+.2f}")

,Spearman correlation with net benefit
Daily queue time,-0.68
Driver's petrol spend (at survey price),+0.30
Driver's net earnings per hour,-0.22
Petrol price,+0.18
Conversion cost,+0.11
CNG price,-0.05
Loan interest rate,-0.02
Working days per year,-0.01
Loan tenor,+0.01


**Conversion cost has a positive sign, which looks backwards.** It doesn't mean dearer kits help. Kit price is
drawn from the keke row for keke drivers (₦100k–650k) and the car row for everyone else (₦300k–1.7m). Keke drivers
in the survey also have small fuel savings, so low kit prices tend to appear in draws that do badly anyway. Looking
within a single vehicle type removes that mix-up, and the correlation turns slightly negative, as expected (check
below). It is small because the loan repayment is small next to fuel savings and queue costs.

In [6]:
for v in ["ride_hailing", "danfo"]:
    g = market[market["vehicle_type"] == v]
    print(f"{v:13s} Spearman(conversion cost, net benefit) = "
          f"{g[['conversion_cost', 'net_benefit']].corr(method='spearman').iloc[0, 1]:+.3f}")

ride_hailing  Spearman(conversion cost, net benefit) = -0.035
danfo         Spearman(conversion cost, net benefit) = -0.051


In [7]:
plot = sensitivity.iloc[::-1]   # plotly draws the first bar at the bottom
fig = go.Figure(go.Bar(
    x=plot.values, y=[INPUT_LABELS[i] for i in plot.index], orientation="h", width=0.6,
    marker=dict(color=[BLUE if v > 0 else ORANGE for v in plot.values], cornerradius=4),
    text=[f"{v:+.2f}" for v in plot.values], textposition="outside", cliponaxis=False,
    textfont=dict(color=INK, size=12),
    hovertemplate="<b>%{y}</b><br>Spearman %{x:+.3f}<extra></extra>",
))
fig.add_vline(x=0, line=dict(color=INK_MUTED, width=1))
fig.update_layout(
    title=dict(text="Queue time moves CNG's net benefit more than any other input<br>"
                    "<sup>Spearman rank correlation with net benefit, 10,000 market-price draws. "
                    "Blue: higher input → higher net benefit. Orange: higher input → lower.</sup>",
               x=0, xanchor="left"),
    template="plotly_white", font=FONT, showlegend=False,
    xaxis=dict(title="Spearman correlation with net benefit", range=[-1, 1], dtick=0.25, gridcolor=GRID,
               zeroline=False),
    yaxis=dict(ticksuffix="  "),
    hoverlabel=dict(bgcolor="white", font=dict(color=INK)),
    width=820, height=460, margin=dict(t=100, l=20, r=40, b=60),
)
fig.show()
fig.write_html(FIG_DIR / "monte_carlo_sensitivity.html", include_plotlyjs="cdn")
print("Saved outputs/figures/monte_carlo_sensitivity.html")

Saved outputs/figures/monte_carlo_sensitivity.html


## 6. What if daily queue time were held at 1, 2, 3 or 4 hours?

Re-run both cases with every draw's daily queue time **fixed** at each value, instead of drawn from the survey.
Everything else is drawn exactly as before, with the same seed. This is a "what if", for example if more stations or
compressors brought queues down to a target.

In [8]:
rows = []
for hours in [1, 2, 3, 4]:
    row = {"daily queue hours": hours}
    for case in ["market", "free"]:
        r = mc.run_simulation(N_DRAWS, SEED, conversion=case, queue_hours_fixed=hours)
        row[f"P(pays), {case}"] = (r["net_benefit"] > 0).mean()
        row[f"median NGN/day, {case}"] = r["net_benefit"].median()
    rows.append(row)
rows.append({"daily queue hours": "as surveyed (drawn)",
             "P(pays), market": (market["net_benefit"] > 0).mean(), "median NGN/day, market": market["net_benefit"].median(),
             "P(pays), free": (free["net_benefit"] > 0).mean(), "median NGN/day, free": free["net_benefit"].median()})
queue_caps = pd.DataFrame(rows).set_index("daily queue hours")
queue_caps.style.format({c: ("{:.0%}" if c.startswith("P") else "{:,.0f}") for c in queue_caps.columns})

,"P(pays), market","median NGN/day, market","P(pays), free","median NGN/day, free"
daily queue hours,,,,
1,98%,"14,284",99%,"16,144"
2,62%,"3,927",66%,"5,709"
3,29%,"-6,583",32%,"-5,160"
4,16%,"-15,020",18%,"-13,380"
as surveyed (drawn),62%,"4,440",66%,"6,141"


## 7. What the results mean

**CNG pays in about 6 draws out of 10, with a wide range of outcomes.** At market kit prices on a loan, CNG comes
out ahead in **62%** of draws, with a median net benefit of about **₦4,400 a day**. The middle 80% of outcomes runs
from a loss of about ₦13,700 to a gain of about ₦21,900 a day. A free conversion only raises the probability to
**66%** (median about ₦6,100). The loan repayment is small, a median of about ₦1,800 per working day, next to fuel
savings and queue costs of tens of thousands of naira.

**Queue time is by far the biggest factor.** Its Spearman correlation with net benefit is **−0.68**, more than twice
the next input. Next come the driver's petrol spend (**+0.30**: drivers who spend more on petrol save more by
switching) and the driver's net earnings per hour (**−0.22**: each hour in a queue costs a higher earner more).
Prices matter less within the ranges in `model_inputs.csv`. Loan terms barely register, and conversion cost's small
positive sign reflects vehicle type (see section 5), not a real benefit from dearer kits.

**Holding queues down changes the picture far more than free kits.** With daily queue time held at **1 hour**,
CNG pays in about **98%** of market-price draws. At **2 hours** it is **62%**, about where it is with surveyed
queues, whose median is 2.25 hours. At **3 hours** it is **29%**, and at **4 hours** **16%**. Going from a 3-hour to a
1-hour daily queue raises the probability by about 70 percentage points; making the conversion free raises it by
about 4. In this model, policy that shortens queues, through more stations, more compressors or longer opening
hours, does far more for drivers than subsidising kits. This holds for drivers like these, whose earnings and
fuel savings are high.

**By vehicle:** ride-hailing drivers come out ahead in about **71%** of draws (median about ₦12,300 a day) and
danfo drivers in about **65%** (about ₦5,100). The keke (3 respondents) and taxi (1 respondent) figures are shown
but marked **insufficient data**: they come from re-using one to three drivers thousands of times.

**Why this is lower than notebook 03**, where 9 of 13 CNG drivers came out ahead: here every driver gets a queue
time drawn at random from the CNG users. That breaks any real link between a driver and their queue, for example a
high earner who refuels where queues are short. Fuel saving is also calculated from prices (the model's method)
rather than from each driver's reported CNG spend.

**Limitations:**
- **Bootstrap from 20 unflagged drivers.** Every simulated driver is one of 20 real ones. The simulation spreads
  their answers across price and queue uncertainty; it does not add new drivers. Results for small groups, and
  extreme outcomes, rest on one or two people.
- **CNG users' survey values applied to petrol drivers.** The 7 petrol drivers get queue times drawn from what CNG
  users reported. They might refuel at different stations or times. Fuel saving uses the model's price ratio rather
  than anything these drivers have experienced.
- **Triangular distributions are assumptions.** The low / mid / high values come from sources, but the triangle
  shape doesn't. It assumes values near "mid" are most likely and that nothing falls outside low–high.
- **No correlation between inputs.** Prices, queue time and loan terms are drawn independently. In reality they may
  move together: for example, higher petrol prices could push more drivers onto CNG and lengthen queues, which would
  make the good-price, short-queue draws too common here.
- **Other assumptions carried over:**
  - Recalled pre-conversion petrol spend is rescaled from ₦1,400/L, assuming the same litres per day.
  - Danfo and taxi drivers use the car conversion price.
  - Earnings are before owner remittance or app commission.
  - The market case covers only the 1–3 years of loan repayment.